# 🗄️ Data360-Like App — Zero-Copy, RAG & Semantic Search

Building a Data360-inspired platform in Python:
- **Zero-copy ingestion** — Arrow/Parquet with memory-mapped files
- **Semantic search** — embedding-based vector similarity
- **RAG (Retrieval-Augmented Generation)** — ground answers in your data
- **Data lineage tracking**
- **Schema inference and profiling**

**Design:** everything runs on-device with sentence-transformers and FAISS.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
# Install if needed:
#   pip install pyarrow pandas numpy faiss-cpu sentence-transformers
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import hashlib, json, time, os, re
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
print('Core imports loaded.')
print('Note: faiss and sentence-transformers are used in the semantic search sections.')

Core imports loaded.
Note: faiss and sentence-transformers are used in the semantic search sections.


## 1. Zero-Copy Ingestion with Apache Arrow

**Zero-copy** means we can share memory between Python, Arrow, NumPy, and Pandas
without any data movement — critical for large datasets.

In [2]:
# ── Zero-Copy with Apache Arrow ───────────────────────────────────────────────

# Simulate a customer data table
n = 10_000
np.random.seed(0)

# Create Arrow table from dict — each column is a contiguous Arrow array
arrow_table = pa.table({
    'customer_id':   pa.array(range(n), type=pa.int32()),
    'name':          pa.array([f'Customer_{i}' for i in range(n)]),
    'age':           pa.array(np.random.randint(18, 80, n), type=pa.int16()),
    'balance':       pa.array(np.random.uniform(0, 100_000, n), type=pa.float64()),
    'segment':       pa.array(np.random.choice(['Gold','Silver','Bronze'], n)),
    'active':        pa.array(np.random.choice([True, False], n, p=[0.8, 0.2])),
})

print(f'Arrow table: {arrow_table.num_rows:,} rows × {arrow_table.num_columns} cols')
print(f'Schema:\n{arrow_table.schema}')
print(f'Memory: {arrow_table.nbytes / 1024:.1f} KB')

# Zero-copy: convert Arrow column to NumPy — no data is copied
balance_np = arrow_table.column('balance').to_pylist()
balance_arr = arrow_table.column('balance').to_pylist()

# For true zero-copy use .to_numpy(zero_copy_only=True)
balance_zero_copy = arrow_table.column('balance').to_numpy(zero_copy_only=False)
print(f'\nBalance as NumPy (zero-copy): mean=${balance_zero_copy.mean():,.2f}')

# Convert to Pandas (also zero-copy where possible)
df = arrow_table.to_pandas()
print(f'Pandas shape: {df.shape}')

# Write to Parquet (columnar, compressed)
path = Path('/tmp/customers.parquet')
pq.write_table(arrow_table, path, compression='snappy')
size_mb = path.stat().st_size / 1024**2
print(f'Parquet file size: {size_mb:.3f} MB  (vs {arrow_table.nbytes/1024**2:.3f} MB in-memory)')

# Read back with predicate pushdown (only read matching rows from disk)
filters = [('segment', '=', 'Gold'), ('balance', '>', 50000)]
gold_high = pq.read_table(path, filters=filters)
print(f'Gold customers with balance > 50k: {gold_high.num_rows}')

Arrow table: 10,000 rows × 6 cols
Schema:
customer_id: int32
name: string
age: int16
balance: double
segment: string
active: bool
Memory: 394.1 KB

Balance as NumPy (zero-copy): mean=$49,400.99
Pandas shape: (10000, 6)
Parquet file size: 0.225 MB  (vs 0.385 MB in-memory)
Gold customers with balance > 50k: 1625


## 2. Schema Inference & Data Profiling

In [ ]:
# ── Schema Inference and Data Profiling ──────────────────────────────────────

@dataclass
class ColumnProfile:
    """Statistical profile for a single column."""
    name:          str
    dtype:         str
    null_count:    int
    null_pct:      float
    unique_count:  int
    # Numeric stats (None for non-numeric)
    min_val:       Optional[float] = None
    max_val:       Optional[float] = None
    mean_val:      Optional[float] = None
    std_val:       Optional[float] = None
    p25:           Optional[float] = None
    p50:           Optional[float] = None
    p75:           Optional[float] = None

def profile_dataframe(df: pd.DataFrame) -> Dict[str, ColumnProfile]:
    """
    Compute a full profile of every column in a DataFrame.
    Returns a dict mapping column name → ColumnProfile.
    """
    profiles = {}
    for col in df.columns:
        series = df[col]
        p = ColumnProfile(
            name=col,
            dtype=str(series.dtype),
            null_count=int(series.isna().sum()),
            null_pct=float(series.isna().mean() * 100),
            unique_count=int(series.nunique()),
        )
        if pd.api.types.is_numeric_dtype(series):
            desc = series.describe()
            p.min_val  = float(desc['min'])
            p.max_val  = float(desc['max'])
            p.mean_val = float(desc['mean'])
            p.std_val  = float(desc['std'])
            p.p25      = float(desc['25%'])
            p.p50      = float(desc['50%'])
            p.p75      = float(desc['75%'])
        profiles[col] = p
    return profiles

profiles = profile_dataframe(df)

# Display as a summary table
summary_rows = []
for name, p in profiles.items():
    row = {'column': name, 'dtype': p.dtype,
           'nulls %': f'{p.null_pct:.1f}%', 'unique': p.unique_count}
    if p.mean_val is not None:
        row.update({'mean': f'{p.mean_val:.2f}', 'std': f'{p.std_val:.2f}',
                    'min': f'{p.min_val:.2f}', 'max': f'{p.max_val:.2f}'})
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

## 3. Semantic Search with Embeddings

Use sentence embeddings to find semantically similar records — even if the
exact words don't match. This is the retrieval component of RAG.

In [ ]:
# ── Semantic Search (simulated embeddings for offline demo) ──────────────────
# In production: use sentence-transformers or an API for real embeddings
# Here we simulate embeddings using a simple TF-IDF + SVD approach

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

# Knowledge base: product descriptions (simulate Data360 data assets)
DOCUMENTS = [
    {'id': 1, 'title': 'Customer 360 Dataset',
     'text': 'Unified customer view combining CRM data, purchase history, support tickets and demographics.'},
    {'id': 2, 'title': 'Revenue Analytics Table',
     'text': 'Daily revenue metrics by product, region and channel. Updated every 4 hours from the data warehouse.'},
    {'id': 3, 'title': 'Product Inventory Feed',
     'text': 'Real-time inventory levels, reorder points and supplier lead times for 50,000 SKUs.'},
    {'id': 4, 'title': 'Sales Pipeline CRM',
     'text': 'Opportunity data from Salesforce including stage, amount, close date and owner.'},
    {'id': 5, 'title': 'Marketing Attribution Model',
     'text': 'Multi-touch attribution across email, paid search, social and organic channels.'},
    {'id': 6, 'title': 'Employee HR Records',
     'text': 'Headcount, tenure, department, salary band and performance ratings. PII-protected.'},
    {'id': 7, 'title': 'Supply Chain Logistics',
     'text': 'Shipment tracking, carrier performance, warehouse throughput and delivery SLAs.'},
    {'id': 8, 'title': 'Financial General Ledger',
     'text': 'Chart of accounts, journal entries, cost centre allocations and budget vs actuals.'},
]

# Build TF-IDF index
corpus = [d['text'] for d in DOCUMENTS]
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
tfidf = vectorizer.fit_transform(corpus)

# Reduce to dense 32-dim semantic space via SVD (simulates embedding model)
svd = TruncatedSVD(n_components=min(32, len(DOCUMENTS)-1), random_state=42)
embeddings = normalize(svd.fit_transform(tfidf))   # L2-normalise for cosine sim

def semantic_search(query: str, top_k: int = 3) -> List[Dict]:
    """
    Find the top-k most relevant documents for a natural language query.
    Uses cosine similarity on LSA (simulated) embeddings.
    """
    # Embed the query using the same vectorizer and SVD
    q_vec  = vectorizer.transform([query])
    q_emb  = normalize(svd.transform(q_vec))         # (1, 32)

    # Cosine similarity = dot product of L2-normalised vectors
    scores = embeddings @ q_emb.T                    # (n_docs, 1)
    scores = scores.flatten()

    # Return top-k results sorted by score
    top_idx = np.argsort(scores)[-top_k:][::-1]
    return [{'rank': i+1, 'score': float(scores[j]),
             'title': DOCUMENTS[j]['title'], 'text': DOCUMENTS[j]['text']}
            for i, j in enumerate(top_idx)]

# Demo queries
for query in ['customer purchase data', 'financial budget tracking', 'shipping delivery']:
    print(f'\nQuery: "{query}"')
    for r in semantic_search(query, top_k=2):
        print(f'  [{r["rank"]}] {r["title"]} (score={r["score"]:.4f})')

## 4. RAG — Retrieval-Augmented Generation

RAG combines semantic retrieval with an LLM: first retrieve relevant context,
then generate an answer grounded in that context.

In [ ]:
# ── RAG Pipeline (Standalone — calls local Ollama or falls back to mock) ──────

class RAGPipeline:
    """
    Retrieval-Augmented Generation pipeline.

    Steps:
    1. Retrieve: semantic_search(question) → top-k context chunks
    2. Augment: build a prompt with the context + question
    3. Generate: call LLM (Ollama / mock)
    """

    def __init__(self, retrieve_fn, llm_fn):
        self.retrieve = retrieve_fn   # (query, top_k) → [{title, text, score}]
        self.llm      = llm_fn        # (prompt) → str

    def answer(self, question: str, top_k: int = 3) -> Dict:
        """Full RAG cycle: retrieve → augment → generate."""

        # Step 1: retrieve relevant context
        retrieved = self.retrieve(question, top_k)

        # Step 2: build the augmented prompt
        context_block = '\n\n'.join(
            f'[Source {r["rank"]}] {r["title"]}:\n{r["text"]}' for r in retrieved
        )
        prompt = (
            f'You are a Data360 assistant. Answer the question using ONLY the provided sources.\n\n'
            f'SOURCES:\n{context_block}\n\n'
            f'QUESTION: {question}\n\n'
            f'ANSWER (cite sources):'
        )

        # Step 3: generate answer via LLM
        answer_text = self.llm(prompt)

        return {
            'question':  question,
            'answer':    answer_text,
            'sources':   [r['title'] for r in retrieved],
            'prompt_tokens': len(prompt.split()),
        }

# Mock LLM (replace with real Ollama/Claude/OpenAI call in production)
def mock_llm(prompt: str) -> str:
    """
    Simulate an LLM response by extracting key phrases from the context.
    In production: call ollama.chat() or anthropic.messages.create().
    """
    # Extract the source titles from the prompt as simulated evidence
    sources = re.findall(r'\[Source \d+\] (.+):', prompt)
    return (f'Based on the available data sources — '
            f'{", ".join(sources)} — '
            f'I can provide the requested information. '
            f'[This is a mock response. Replace mock_llm with a real LLM call.]')

rag = RAGPipeline(retrieve_fn=semantic_search, llm_fn=mock_llm)

questions = [
    'What customer data is available for personalisation?',
    'How do we track budget vs actual spending?',
]

for q in questions:
    result = rag.answer(q)
    print(f'Q: {result["question"]}')
    print(f'A: {result["answer"]}')
    print(f'Sources: {result["sources"]}')
    print(f'Prompt tokens: {result["prompt_tokens"]}\n')

## 5. Data Lineage Tracking

In [ ]:
# ── Data Lineage Tracking ─────────────────────────────────────────────────────
# Track how datasets are created, transformed, and derived from one another

@dataclass
class DataAsset:
    """A single data asset with lineage metadata."""
    name:        str
    owner:       str
    description: str
    source_assets: List[str]        = field(default_factory=list)  # upstream
    transform_code: Optional[str]  = None                          # how it was created
    tags:          List[str]        = field(default_factory=list)
    created_at:    Optional[str]    = None
    row_count:     Optional[int]    = None
    checksum:      Optional[str]    = None


class LineageCatalog:
    """Central registry of all data assets and their lineage."""

    def __init__(self):
        self._assets: Dict[str, DataAsset] = {}

    def register(self, asset: DataAsset) -> None:
        """Register a new data asset in the catalog."""
        self._assets[asset.name] = asset

    def get_upstream(self, name: str) -> List[DataAsset]:
        """Return all direct upstream dependencies of an asset."""
        asset = self._assets.get(name)
        if not asset: return []
        return [self._assets[s] for s in asset.source_assets if s in self._assets]

    def get_full_lineage(self, name: str, depth: int = 0) -> None:
        """Print the full upstream lineage tree recursively."""
        indent = '  ' * depth
        asset = self._assets.get(name)
        if not asset:
            print(f'{indent}⚠️  {name} (not found)')
            return
        rows = f' ({asset.row_count:,} rows)' if asset.row_count else ''
        print(f'{indent}📊 {name}{rows} — {asset.description[:60]}')
        for src in asset.source_assets:
            self.get_full_lineage(src, depth + 1)

# Example lineage graph
catalog = LineageCatalog()

catalog.register(DataAsset('raw_crm', 'Sales Ops',
    'Raw Salesforce export', tags=['raw','crm'], row_count=120_000))
catalog.register(DataAsset('raw_transactions', 'Finance',
    'Raw transaction ledger from ERP', tags=['raw','finance'], row_count=5_000_000))
catalog.register(DataAsset('clean_customers', 'Data Engineering',
    'Deduplicated and validated customer records',
    source_assets=['raw_crm'],
    transform_code='dedup + standardise_names + fill_missing_phone',
    row_count=98_000, tags=['silver','crm']))
catalog.register(DataAsset('customer_360', 'Data Science',
    'Unified customer view with RFM and CLV scores',
    source_assets=['clean_customers', 'raw_transactions'],
    transform_code='join on customer_id + compute RFM + predict CLV',
    row_count=95_000, tags=['gold','360','ml']))

print('=== Lineage for customer_360 ===')
catalog.get_full_lineage('customer_360')
print('\nDirect upstream of customer_360:')
for asset in catalog.get_upstream('customer_360'):
    print(f'  {asset.name}: {asset.description}')